In [0]:
df = spark.table("bankingdata.bronze.card")

In [0]:
df.columns

### Handling rescue data column

In [0]:
from pyspark.sql.functions import *

In [0]:
df_rescued = df.filter(col("_rescued_data").isNotNull())
df = df.filter(col("_rescued_data").isNull())

In [0]:
df_rescued_final = df_rescued.select(
    col("CardID"),          
    col("_rescued_data"),       
    col("source_file"),        
    col("ingestion_time")       
)

In [0]:
df_rescued_final.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("bankingdata.rescueddata.card_rescued")

### Schema

In [0]:
df = df.select(
    col("CardID").cast("int").alias("CardID"),
    col("AccountID").cast("int").alias("AccountID"),
    col("CardType").cast("string").alias("CardType"),
    col("CardLimit").cast("double").alias("CardLimit"),
    col("ExpiryDate").cast("date").alias("ExpiryDate"),
    col("ModifiedDate").cast("timestamp").alias("ModifiedDate")
)

### Primary Key Validation

In [0]:
df = df.filter(
    col("CardID").isNotNull() &
    (trim(col("CardID").cast("string")) != "") &
    (col("CardID") > 0)
)

### Business Quality Check


In [0]:
df = df.filter(
    # AccountID (FK)
    col("AccountID").isNotNull() &
    (col("AccountID") > 0) &

    # CardType
    col("CardType").isNotNull() &
    col("CardType").isin("Debit", "Credit") &

    # CardLimit
    col("CardLimit").isNotNull() &
    (col("CardLimit") >= 0) &

    # ExpiryDate
    col("ExpiryDate").isNotNull() &
    (col("ExpiryDate") >= current_date()) &

    # ModifiedDate
    col("ModifiedDate").isNotNull() &
    (col("ModifiedDate") <= current_timestamp())
)

### Remove Duplicates

In [0]:
from pyspark.sql.window import Window

window_spec = Window.partitionBy("CardID") \
                    .orderBy(col("ModifiedDate").desc())

df = df.withColumn("rn", row_number().over(window_spec)) \
                   .filter(col("rn") == 1) \
                   .drop("rn")

### Delta table

In [0]:
# save table
silver_table = "bankingdata.silver.card"

df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(silver_table)

print("Silver table created:", silver_table)